In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Import des librairies utiles ( ma caisse à outils)

from bs4 import BeautifulSoup
from tensorflow.keras import layers
import tensorflow as tf
import keras
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn import preprocessing
from sklearn.model_selection import TimeSeriesSplit
from keras.models import Sequential
from keras.models import load_model
from keras.layers import LSTM, Dense,Dropout
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense, Bidirectional, TimeDistributed, RepeatVector, Flatten
from keras.callbacks import EarlyStopping

In [ ]:
# le fichier csv avec les 5 premiers numeros depuis 1976

df=pd.read_csv('/kaggle/input/d/jeanyvessimon/loto-chance-2008-2025/loto_chance_2008_2025(1).csv', parse_dates=["date_de_tirage"])

In [ ]:
df

In [ ]:
# le fichier csv uniquement les numeros chances

# df=pd.read_csv('/kaggle/input/loto-2008-2024-num-chance/loto_2008_2024_num_chance.csv')

In [ ]:
# set_index('date_de_tirage') 
# Cette méthode est utilisée pour définir une colonne existante comme l'index du DataFrame

# df = df.set_index('date_de_tirage')
# Cette ligne de code remplace le DataFrame df par une nouvelle version 
# où la colonne date_de_tirage est utilisée comme index.


df = df.set_index('date_de_tirage')

In [ ]:
df

In [ ]:
# inversion du dataframe pour placer le dernier tirage en dernière position

df = df.iloc[::-1]


In [ ]:
# je supprime le dernier tirage pour le predire

df = df.drop(df.index[-1])


In [ ]:
# sélection ou extractions des colonnes à  traiter

df = df[['boule_2']]


In [ ]:
df

/*************************************************************

***********************************************************

/***********************************************************

In [ ]:
# Pour Supprimer toutes les lignes de la colonne 'boule_2' qui est égale à 37 par exemple

#df = df[df['boule_2'] != 37]

In [ ]:
# pour Supprimer les numeros deja sorties. 
# à partir de la boule2

# df = df[df['boule_3'] != 37]
# df = df[df['boule_3'] != 31]
# df = df[df['boule_4'] != ]
# df = df[df['boule_5'] != 22 ]



In [ ]:
#from sklearn.preprocessing import MinMaxScaler
#scaler = MinMaxScaler().fit(df.values)
#transformed_dataset = scaler.transform(df.values)
#transformed_df = pd.DataFrame(data=transformed_dataset, index=df.index)
#transformed_df

In [ ]:
#transformed_df=pd.DataFrame(df.values, index=df.index)

In [ ]:
#transformed_df

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Exemple de DataFrame
#data = {'A': [10, 20, 30, 40, 50],
        #'B': [1, 2, 3, 4, 5]}
#df = pd.DataFrame(data)

# Initialisation du scaler Min-Max
#scaler = MinMaxScaler()

# Normalisation du DataFrame
#transformed_dataset = scaler.transform(df.values)
#transformed_df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

#transformed_df = pd.DataFrame(data=transformed_dataset, index=df.index)

#print("DataFrame normalisé :")
#print(transformed_df)

In [ ]:
scaler = StandardScaler().fit(df.values)
transformed_dataset = scaler.transform(df.values)
transformed_df = pd.DataFrame(data=transformed_dataset, index=df.index)

number_of_rows= df.values.shape[0] # toutes les lignes
window_length = 17 # longueur des fenetres
number_of_features = df.values.shape[1] # caracteristiques

train = np.empty([number_of_rows-window_length, window_length, number_of_features], dtype=float)
label = np.empty([number_of_rows-window_length, number_of_features], dtype=float)

for i in range(0, number_of_rows-window_length):
    train[i]=transformed_df.iloc[i:i+window_length, 0: number_of_features]
    label[i]=transformed_df.iloc[i+window_length: i+window_length+1, 0: number_of_features]

In [ ]:
transformed_df

In [ ]:
# exemple de LSTM


model= Sequential()
model.add(LSTM(200, input_shape=(window_length, number_of_features), return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(200, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(200, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(200, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(50, return_sequences=False))
model.add(Dense(number_of_features))

model.compile(loss='mse', optimizer='Adam', metrics=['accuracy'])
#model.compile(
    #optimizer=keras.optimizers.Adam, loss="mse", metrics=['mae']
#)

history=model.fit(train, label, batch_size=64, epochs=400, verbose=0)

'''
# Initialising the RNN
model = Sequential()
# Adding the input layer and the LSTM layer
model.add(Bidirectional(LSTM(240,
                        input_shape = (window_length, number_of_features),
                        return_sequences = True)))
model.add(Dropout(0.2))
# Adding a second LSTM layer
model.add(Bidirectional(LSTM(240,
                        input_shape = (window_length, number_of_features),
                        return_sequences = True)))
model.add(Dropout(0.2))
model.add(Bidirectional(LSTM(240,
                        input_shape = (window_length, number_of_features),
                        return_sequences = False)))
model.add(Dense(number_of_features))


model.compile(loss='mse', optimizer='adam', metrics=['accuracy'])

history=model.fit(train, label, batch_size=64, epochs=200, verbose=0)
'''
'''
# transformed_df.iloc[-window_length:, :]
# Cette partie du code utilise la méthode iloc pour sélectionner
# les window_length dernières lignes du DataFrame transformed_df.
# Le : après -window_length: signifie que toutes les colonnes sont sélectionnées.
# .values : Cette méthode convertit le DataFrame sélectionné en un tableau NumPy.

# last_window = transformed_df.iloc[-window_length:, :].values

# Cette ligne redimensionne le tableau last_window pour qu'il ait la forme (1, window_length, number_of_features).
# last_window = last_window.reshape((1, window_length, number_of_features))

# Cette ligne utilise le modèle pour faire une prédiction sur la dernière fenêtre de données.
# scaled_next_row = model.predict(last_windows)


'''




In [ ]:
model.save("my_boule_2.keras")

In [ ]:
# model LSTM

'''

model = Sequential()
model.add(LSTM(64,input_shape=(window_length, number_of_features),return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(64, return_sequences=True))
model.add(LSTM(64,return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(number_of_features))

# choisir le model
model.compile(loss='mse', optimizer='adam')
#, metrics=['acc'])
# model.compile(loss='mean_squared_error', optimizer='adam', metrics=['accuracy'])

# Entrainement.....
history= model.fit(train, label,batch_size=64, epochs=300, verbose=0)
'''
'''

'''
'''


# LSTM Bidirectional

model = Sequential()
model.add(Bidirectional(LSTM(100, dropout=0.2, return_sequences=True), input_shape=(window_length, number_of_features)))
model.add(LSTM(100, return_sequences=True))
model.add(LSTM(64, activation='relu',dropout=0.2))
model.add(Dense(number_of_features))

model.compile(loss='mse', optimizer='adam', metrics=['accuracy'])


# Entrainement.....
history= model.fit(train, label,batch_size=64, epochs=400, verbose=1)
'''
'''

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

# Construction du modèle GRU
model = Sequential()
model.add(GRU(80, input_shape=(window_length, number_of_features),return_sequences=True))
model.add(Dropout(0.2))
model.add(GRU(80))
model.add(Dropout(0.2))
model.add(Dense(number_of_features))

model.compile(loss='mse', optimizer='adam')

history= model.fit(train, label,batch_size=64, epochs=350, verbose=0)
'''
'''

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

# Construction du modèle GRU
model = Sequential()
model.add(GRU(50, return_sequences=True, input_shape=(window_length, number_of_features),return_sequences=True))
model.add(GRU(50))
model.add(Dense(1=number_of_features))
model.compile(loss='mse', optimizer='adam')
'''
'''
model = Sequential()
model.add(LSTM(100, input_shape=(window_length, number_of_features), return_sequences=True))
model.add(LSTM(50, return_sequences=False))
model.add(RepeatVector(window_length))
model.add(LSTM(100, dropout=0.1, return_sequences=True))
model.add(LSTM(50, return_sequences=True))
model.add(TimeDistributed(Dense(number_of_features)))
model.add(Flatten())
model.add(Dense(number_of_features))

model.compile(loss='mse', optimizer='adam') #, metrics=['acc'])

# Entrainement.....
history= model.fit(train, label,batch_size=64, epochs=400)

# sauvegarde du model avec son nom de colonne....
# # sauvegarde du model avec son nom de colonne....
model.save('input/model_boule_1.h5')
model.save('input/model_boule_2.h5')
model.save('input/model_boule_3.h5')
model.save('input/model_boule_4.h5')
model.save('input/model_boule_5.h5')
model.save('input/model_numero_chance.h5')

'''

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['accuracy'])
plt.legend(['train_loss'])
plt.show()

In [ ]:
##### to_predict=df.iloc[0:12] # sélectionne les x dernières lignes de votre DataFrame
to_predict=df.tail(17)
scaled_to_predict = scaler.transform(to_predict)

# predictions

scaled_predicted_output_1 = model.predict(np.array([scaled_to_predict]))
data = scaler.inverse_transform(scaled_predicted_output_1).astype(int)
df_predict = pd.DataFrame(data)
#df_predict = pd.DataFrame(data, columns=['boule_1'])
#df.to_csv(''+filename+'.csv', index=False)
df_predict

In [ ]:
36 	1 	45 	22 	25

12 	24 	6 	16 	2

In [ ]:
49 	12 	42 	29 	13

In [ ]:
to_predict

In [ ]:
reconstructed_model = keras.models.load_model('/kaggle/input/my_model/keras/default/1/my_model.keras')

In [ ]:
2025-05-12 	12 	24 	6 	16 	2

In [ ]:
to_predict

In [ ]:
2025-01-01 	8 	11 	2 	25 	21

In [ ]:
from bs4 import BeautifulSoup
from tensorflow.keras import layers
import tensorflow as tf
import keras
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn import preprocessing
from sklearn.model_selection import TimeSeriesSplit
from keras.models import Sequential
from keras.models import load_model
from keras.layers import LSTM, Dense,Dropout
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense, Bidirectional, TimeDistributed, RepeatVector, Flatten
from keras.callbacks import EarlyStopping

In [ ]:
# Charger les données
df = pd.read_csv("/kaggle/input/loto-2008-2025/loto_2008_2025.csv")
df = df.set_index('date_de_tirage')

In [ ]:
df

In [ ]:
#je supprime le dernier tirage pour le predire

df = df.drop(df.index[-1])

In [ ]:
df

In [ ]:

# Charger les données
#df = pd.read_csv("/kaggle/input/loto-2008-2025/loto_2008_2025.csv")
#df = df.set_index('date_de_tirage')
#je supprime le dernier tirage pour le predire

#df = df.drop(df.index[-1])

# Fusionner toutes les colonnes de numéros dans une seule colonne pour comptabiliser les fréquences
numeros = pd.concat([df['boule_1'], df['boule_2'], df['boule_3'], df['boule_4'], df['boule_5']])#, ignore_index=True)
#etoiles = pd.concat([df['Etoile1'], df['Etoile2']], ignore_index=True)
#numeros = df[['boule_2']]

# Calculer les fréquences des numéros principaux et des étoiles
frequence_numeros = numeros.value_counts().sort_values(ascending=False)
#frequence_etoiles = etoiles.value_counts().sort_values(ascending=False)

# Afficher les résultats
print("Fréquence des numéros principaux :\n", frequence_numeros)
#print("\nFréquence des étoiles :\n", frequence_etoiles)

# Extraire les 5 numéros et 2 étoiles les plus fréquents pour une grille recommandée
meilleurs_numeros = frequence_numeros.head(1).index.tolist()
#meilleures_etoiles = frequence_etoiles.head(2).index.tolist()

print("\nGrille recommandée :")
print("Numéros : ", meilleurs_numeros)
#print("Étoiles : ", meilleures_etoiles

In [ ]:
# pour Supprimer les numeros deja sorties. 
# à partir de la boule2

df = df[df['boule_3'] != 23]
# df = df[df['boule_3'] != 31]
# df = df[df['boule_4'] != ]
# df = df[df['boule_5'] != 22 ]


In [ ]:
df

In [ ]:
import pandas as pd
import numpy as np

from sklearn import metrics
from sklearn import preprocessing
import tensorflow as tf

import matplotlib.pyplot as plt

In [ ]:
# Load the CSV file into a DataFrame
df = pd.read_csv("/kaggle/input/d/jeanyvessimon/loto-1976-2025/loto_1976_2025(3).csv")


In [ ]:
print(df.shape)
df.head()

In [ ]:
def mean_absolute_percentage_error_func(y_true, y_pred):
    '''
    Calculate the mean absolute percentage error as a metric for evaluation
    
    Args:
        y_true (float64): Y values for the dependent variable (test part), numpy array of floats 
        y_pred (float64): Predicted values for the dependen variable (test parrt), numpy array of floats
    
    Returns:
        Mean absolute percentage error 
    '''    
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    

In [ ]:
def timeseries_evaluation_metrics_func(y_true, y_pred):
    '''
    Calculate the following evaluation metrics:
        - MSE
        - MAE
        - RMSE
        - MAPE
        - R²
    
    Args:
        y_true (float64): Y values for the dependent variable (test part), numpy array of floats 
        y_pred (float64): Predicted values for the dependen variable (test parrt), numpy array of floats
    
    Returns:
        MSE, MAE, RMSE, MAPE and R² 
    '''    
    print('Evaluation metric results: ')
    print(f'MSE is : {metrics.mean_squared_error(y_true, y_pred)}')
    print(f'MAE is : {metrics.mean_absolute_error(y_true, y_pred)}')
    print(f'RMSE is : {np.sqrt(metrics.mean_squared_error(y_true, y_pred))}')
    print(f'MAPE is : {mean_absolute_percentage_error_func(y_true, y_pred)}')
    print(f'R2 is : {metrics.r2_score(y_true, y_pred)}',end='\n\n')



In [ ]:
def univariate_data_prep_func(dataset, start, end, window, horizon):
    '''
    Prepare univariate data that is suitable for a time series
    
    Args:
        dataset (float64): Scaled values for the dependent variable, numpy array of floats 
        start (int): Start point of range, integer
        end (int): End point of range, integer
        window (int): Number of units to be viewed per step, integer
        horizon (int): Number of units to be predicted, integer
    
    Returns:
        X (float64): Generated X-values for each step, numpy array of floats
        y (float64): Generated y-values for each step, numpy array of floats
    '''   
    X = []
    y = []

    start = start + window
    if end is None:
        end = len(dataset) - horizon

    for i in range(start, end):
        indicesx = range(i-window, i)
        X.append(np.reshape(dataset[indicesx], (window, 1)))
        indicesy = range(i,i+horizon)
        y.append(dataset[indicesy])
    return np.array(X), np.array(y)

In [ ]:
#df = df.drop_duplicates(subset=['date_de_tirage'], keep=False)

#df.shape

In [ ]:
test_data = df['boule_5'].tail(10)

df = df.drop(df['boule_2'].tail(10).index)

df.shape

In [ ]:
uni_data = df['boule_5']
uni_data.index = df['date_de_tirage']
uni_data.head()

In [ ]:
uni_data = uni_data.values
scaler_x = preprocessing.MinMaxScaler()
x_scaled = scaler_x.fit_transform(uni_data.reshape(-1, 1))

In [ ]:
univar_hist_window_sss = 48
horizon_sss = 1
# 35120 observations in total
# 30000 should be part of the training (5120 validation)
train_split_sss = 3000

x_train_uni_sss, y_train_uni_sss = univariate_data_prep_func(x_scaled, 0, train_split_sss, 
                                                             univar_hist_window_sss, horizon_sss)

x_val_uni_sss, y_val_uni_sss = univariate_data_prep_func(x_scaled, train_split_sss, None, 
                                                         univar_hist_window_sss, horizon_sss)



In [ ]:
print ('Length of first Single Window:')
print (len(x_train_uni_sss[0]))
print()
print ('Target horizon:')
print (y_train_uni_sss[0])

In [ ]:
univar_hist_window_hs = 48
horizon_hs = 10
# see comments of section 4.5.1
train_split_hs = 3000

x_train_uni_hs, y_train_uni_hs = univariate_data_prep_func(x_scaled, 0, train_split_hs, 
                                                           univar_hist_window_hs, horizon_hs)

x_val_uni_hs, y_val_uni_hs = univariate_data_prep_func(x_scaled, train_split_hs, None, 
                                                       univar_hist_window_hs, horizon_hs)

In [ ]:
print ('Length of first Single Window:')
print (len(x_train_uni_hs[0]))
print()
print ('Target horizon:')
print (y_train_uni_hs[0])

In [ ]:
BATCH_SIZE_sss = 256
BUFFER_SIZE_sss = 150

train_univariate_sss = tf.data.Dataset.from_tensor_slices((x_train_uni_sss, y_train_uni_sss))
train_univariate_sss = train_univariate_sss.cache().shuffle(BUFFER_SIZE_sss).batch(BATCH_SIZE_sss).repeat()

validation_univariate_sss = tf.data.Dataset.from_tensor_slices((x_val_uni_sss, y_val_uni_sss))
validation_univariate_sss = validation_univariate_sss.batch(BATCH_SIZE_sss).repeat()

In [ ]:
BATCH_SIZE_hs = 256
BUFFER_SIZE_hs = 150

train_univariate_hs = tf.data.Dataset.from_tensor_slices((x_train_uni_hs, y_train_uni_hs))
train_univariate_hs = train_univariate_hs.cache().shuffle(BUFFER_SIZE_hs).batch(BATCH_SIZE_hs).repeat()

validation_univariate_hs = tf.data.Dataset.from_tensor_slices((x_val_uni_hs, y_val_uni_hs))
validation_univariate_hs = validation_univariate_hs.batch(BATCH_SIZE_hs).repeat()

In [ ]:
n_steps_per_epoch = 117
n_validation_steps = 20
n_epochs = 110

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(100, input_shape=x_train_uni_sss.shape[-2:],return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(100,return_sequences=False),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(units=horizon_sss)])


In [ ]:
model.compile(loss='mse',
              optimizer='adam')

In [ ]:
# Sauvegarder le modèle
#model_path = '/kaggle/input/lstm_model/keras/default/1/lstm_model_sss.h5'
#model.save(model_path)
model.save('/kaggle/working/my_model.keras') 
#model.save("my_model.keras")

In [ ]:
model_path = ("/kaggle/working/my_model.keras")

In [ ]:
keras_callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', 
                                                    min_delta=0, patience=10, 
                                                    verbose=1, mode='min'),
                   tf.keras.callbacks.ModelCheckpoint(model_path,monitor='val_loss', 
                                                      save_best_only=True, 
                                                      mode='min', verbose=0)]

In [ ]:
history = model.fit(train_univariate_sss,
                    epochs=n_epochs,
                    steps_per_epoch=n_steps_per_epoch,
                    validation_data=validation_univariate_sss,
                    validation_steps=n_validation_steps,
                    verbose =0)
                    

In [ ]:
model.save('/kaggle/working/my_model.keras') 

In [ ]:
#model_path = ("/kaggle/working/my_model.keras")

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(loss) + 1)


plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.models import load_model
# Charger le modèle
trained_lstm_model_sss = tf.keras.models.load_model(model_path)

In [ ]:
df_temp = df['boule_5']
test_horizon = df_temp.tail(univar_hist_window_sss)
test_history = test_horizon.values

result = []
# Define Forecast length here
window_len = len(test_data)
test_scaled = scaler_x.fit_transform(test_history.reshape(-1, 1))

for i in range(1, window_len+1): 
    test_scaled = test_scaled.reshape((1, test_scaled.shape[0], 1))
    # Inserting the model
    predicted_results = trained_lstm_model_sss.predict(test_scaled)
    
    print(f'predicted : {predicted_results}')
    result.append(predicted_results[0])
    test_scaled = np.append(test_scaled[:,1:],[[predicted_results]])

In [ ]:
result_inv_trans = scaler_x.inverse_transform(result)
result_inv_trans

In [ ]:
timeseries_evaluation_metrics_func(test_data, result_inv_trans)

In [ ]:
rmse_lstm_model_sss = np.sqrt(metrics.mean_squared_error(test_data, result_inv_trans))

In [ ]:
plt.plot(list(test_data))
plt.plot(list(result_inv_trans))
plt.title("Actual vs Prediction")
plt.ylabel("boule_1")
plt.legend(('Actual','prediction'))
plt.grid(True)
plt.show()

In [ ]:
test_data = df['boule_5'].tail(10)

In [ ]:
rmse_lstm_model_sss

In [ ]:
test_data

In [ ]:
result_inv_trans

In [ ]:
fin

In [ ]:
# Load the CSV file into a DataFrame
frequency_data = pd.read_csv("/kaggle/input/d/jeanyvessimon/loto-1976-2025/loto_1976_2025(2).csv")


In [ ]:
frequency_data

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

# Chargement et prétraitement des données
data = pd.read_csv('/kaggle/input/d/jeanyvessimon/loto-1976-2025/loto_1976_2025(1).csv')
numbers = data['boule_2'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(-1, 1))
numbers_normalized = scaler.fit_transform(numbers)

# Fonction pour créer des séquences
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

# Fonction pour construire et entraîner le modèle
def build_and_train_model(seq_length, X_train, y_train, X_val, y_val):
    model = Sequential()
    model.add(GRU(50, return_sequences=True, input_shape=(seq_length, 1)))
    model.add(GRU(50))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X_train, y_train, epochs=50, batch_size=64, validation_data=(X_val, y_val), verbose=0)
    return model

# Validation croisée
tscv = TimeSeriesSplit(n_splits=5)
seq_lengths = [5, 10, 15, 20, 25]
best_seq_length = None
best_val_loss = float('inf')

for seq_length in seq_lengths:
    val_losses = []
    for train_index, val_index in tscv.split(numbers_normalized):
        X_train, y_train = create_sequences(numbers_normalized[train_index], seq_length)
        X_val, y_val = create_sequences(numbers_normalized[val_index], seq_length)
        X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
        X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))
        model = build_and_train_model(seq_length, X_train, y_train, X_val, y_val)
        val_loss = model.evaluate(X_val, y_val, verbose=0)
        val_losses.append(val_loss)
    avg_val_loss = np.mean(val_losses)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_seq_length = seq_length

print(f"Best sequence length: {best_seq_length} with validation loss: {best_val_loss}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Génération d'une suite de chiffres aléatoires
#np.random.seed(42)  # Pour la reproductibilité
#numbers = np.random.randint(0, 10, size=100).reshape(-1, 1)
data= pd.read_csv('/kaggle/input/d/jeanyvessimon/loto-1976-2025/loto_1976_2025(1).csv')

# je supprime le dernier tirage pour le predire
data = data.drop(data.index[-1])

numbers = data['boule_2'].values.reshape(-1, 1)

# Normalisation des données entre 0 et 1
#scaler = MinMaxScaler(feature_range=(0, 1))
#numbers_normalized = scaler.fit_transform(numbers)

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
number_normalized = scaler.fit_transform(numbers)


# Création des séquences de données
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 12
X, y = create_sequences(numbers_normalized, seq_length)

# Redimensionnement des données pour LSTM
X = X.reshape((X.shape[0], X.shape[1], 1))


In [ ]:
# Construction du modèle LSTM
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(seq_length, 1)))
model.add(Dropout(0.2))
model.add(LSTM(64, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

# Entraînement du modèle
history=model.fit(X, y, epochs=400, batch_size=64, verbose=0)


In [ ]:
plt.plot(history.history['loss'])
plt.legend(['train_loss'])
plt.show()

In [ ]:
# Fonction pour prédire le prochain nombre
seq_length=1

def predict_next_number(model, data, seq_length):
    input_seq = data[-seq_length:]
    input_seq = input_seq.reshape((1, seq_length, 1))
    predicted = model.predict(input_seq)
    return predicted[0, 0]

# Prévision du prochain nombre
next_number_normalized = predict_next_number(model, numbers_normalized, seq_length)

# Dénormalisation du prochain nombre
next_number = scaler.inverse_transform(next_number_normalized.reshape(-1, 1))

print(f"Le prochain nombre prédit est : {next_number[0, 0]}")


In [ ]:
# Ajout du prochain nombre prédit à la série temporelle originale
numbers_with_prediction = np.append(numbers, next_number)

# Visualisation des résultats
plt.plot(numbers, label='True Data')
plt.plot(len(numbers), next_number, label='Predicted Next Number', marker='o')
plt.legend()
plt.show()


# debut

In [ ]:
import pandas as pd
#dataframe = pd.read_csv('./GOOGL.csv', index_col="Date", parse_dates=True)
dataframe = pd.read_csv('/kaggle/input/d/jeanyvessimon/loto-1976-2025/loto_1976_2025(3).csv')

In [ ]:
import math
import matplotlib.pyplot as plt
import keras
import pandas as pd
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout
from keras.layers import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping

In [ ]:
df=pd.read_csv("/kaggle/input/d/jeanyvessimon/loto-1976-2025/loto_1976_2025(3).csv")

df.head()

In [ ]:
# set_index('date_de_tirage') 
# Cette méthode est utilisée pour définir une colonne existante comme l'index du DataFrame

dataframe = dataframe.set_index('date_de_tirage')


In [ ]:
training_set = df.iloc[:800, 1:2].values
test_set = df.iloc[800:, 1:2].values

In [ ]:
# Feature Scaling
sc = MinMaxScaler(feature_range = (0, 1))
training_set_scaled = sc.fit_transform(training_set)

In [ ]:
# Creating a data structure with 60 time-steps and 1 output
X_train = []
y_train = []
for i in range(60, 800):
    X_train.append(training_set_scaled[i-60:i, 0])
    y_train.append(training_set_scaled[i, 0])
X_train, y_train = np.array(X_train), np.array(y_train)

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [ ]:
model = Sequential()

#Adding the first LSTM layer and some Dropout regularisation
model.add(LSTM(units = 50, return_sequences = True, input_shape = (X_train.shape[1], 1)))
model.add(Dropout(0.2))

# Adding a second LSTM layer and some Dropout regularisation
model.add(LSTM(units = 50, return_sequences = True))
model.add(Dropout(0.2))

# Adding a third LSTM layer and some Dropout regularisation
model.add(LSTM(units = 50, return_sequences = True))
model.add(Dropout(0.2))

# Adding a fourth LSTM layer and some Dropout regularisation
model.add(LSTM(units = 50))
model.add(Dropout(0.2))

# Adding the output layer
model.add(Dense(units = 1))

In [ ]:
# Compiling the RNN
model.compile(optimizer = 'adam', loss = 'mean_squared_error')

In [ ]:
# Fitting the RNN to the Training set
model.fit(X_train, y_train, epochs = 100, batch_size = 32)



In [ ]:
# Getting the predicted stock price of 2017
dataset_train = df.iloc[:800, 1:2]
dataset_test = df.iloc[800:, 1:2]

In [ ]:
dataset_total = pd.concat((dataset_train, dataset_test), axis = 0)

In [ ]:
inputs = dataset_total[len(dataset_total) - len(dataset_test) - 60:].values

In [ ]:
inputs = inputs.reshape(-1,1)
inputs = sc.transform(inputs)
X_test = []
for i in range(60, 519):
    X_test.append(inputs[i-60:i, 0])
X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

print(X_test.shape)
# (459, 60, 1)

In [ ]:
predicted_stock_price = model.predict(X_test)
predicted_stock_price = sc.inverse_transform(predicted_stock_price)

In [ ]:
# Visualising the results
plt.plot(df.loc[800:, 'boule_1'],dataset_test.values, color = 'red', label = 'Real TESLA Stock Price')
plt.plot(df.loc[800:, 'boule_1'],predicted_stock_price, color = 'blue', label = 'Predicted TESLA Stock Price')
plt.xticks(np.arange(0,459,50))
plt.title('TESLA Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('TESLA Stock Price')
plt.legend()
plt.show()

In [ ]:
# Make sure that you have all these libaries available to run the code successfully
from pandas_datareader import data
import matplotlib.pyplot as plt
import pandas as pd
import datetime as dt
import urllib.request, json
import os
import numpy as np
import tensorflow as tf # This code has been tested with TensorFlow 1.6
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv('/kaggle/input/euromillions-2004-2025/euromillions_2004_2025.csv')

In [ ]:
# Sort DataFrame by date
df = df.sort_values('date_de_tirage')

# Double check the result
df.head()


In [ ]:
plt.figure(figsize = (18,9))
plt.plot(range(df.shape[0]),(df['boule_1']+df['boule_2'])/2.0)
plt.xticks(range(0,df.shape[0],500),df['date_de_tirage'].loc[::500],rotation=45)
plt.xlabel('Date',fontsize=18)
plt.ylabel('Mid Price',fontsize=18)
plt.show()


In [ ]:
# First calculate the mid prices from the highest and lowest
high_prices = df.loc[:,'boule_1']
low_prices = df.loc[:,'boule_2']
mid_prices = (high_prices+low_prices)/2.0


In [ ]:
mid_prices

In [ ]:
train_data = mid_prices[:1000]
test_data = mid_prices[10000:]


In [ ]:
from bs4 import BeautifulSoup
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, TimeDistributed, RepeatVector, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
df=pd.read_csv("/kaggle/input/loto-2008-2025/loto_2008_2025.csv")

In [ ]:
# Additional utility functions for feature engineering
def is_under(data, number):
    # Check if each number is less than or equal to the given number
    return ((data['boule_1'] <= number).astype(int) +
            (data['boule_3'] <= number).astype(int) +
            (data['boule_3'] <= number).astype(int) +
            (data['boule_4'] <= number).astype(int) +
            (data['boule_5'] <= number).astype(int))

def is_pair(data):
    # Check if each number is an even number
    return ((data['boule_1'].isin(pairs)).astype(int) +
            (data['boule_2'].isin(pairs)).astype(int) +
            (data['boule_3'].isin(pairs)).astype(int) +
            (data['boule_4'].isin(pairs)).astype(int) +
            (data['boule_5'].isin(pairs)).astype(int))
def is_impair(data):
    # Check if each number is an odd number
    return ((data['boule_1'].isin(impairs)).astype(int) +
            (data['boule_2'].isin(impairs)).astype(int) +
            (data['boule_3'].isin(impairs)).astype(int) +
            (data['boule_4'].isin(impairs)).astype(int) +
            (data['boule_5'].isin(impairs)).astype(int))

def is_pair_etoile(data):
    # Check if the chance number is an even number
    return (data['numero_chance'].isin(pairs)).astype(int)

def is_impair_etoile(data):
    # Check if the chance number is an odd number
    return (data['numero_chance'].isin(impairs)).astype(int)

def sum_diff(data):
    # Calculate the sum of the squared differences between consecutive numbers
    return ((data['boule_1'] - data['boule_1']) ** 2 +
            (data['boule_2'] - data['boule_2']) ** 2 +
            (data['boule_3'] - data['boule_3']) ** 2 +
            (data['boule_4'] - data['boule_4']) ** 2)

def freq_val(data, column):
    # Calculate the frequency of each number up to the current position
    tab = data[column].values.tolist()
    freqs = []
    pos = 1
    for e in tab:
        freqs.append(tab[0:pos].count(e))
        pos = pos + 1
    return freqs

# New feature engineering functions
def calculate_mean(data):
    # Calculate the mean of the lottery numbers
    return data[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']].mean(axis=1)

def calculate_median(data):
    # Calculate the median of the lottery numbers
    return data[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']].median(axis=1)

def calculate_std(data):
    # Calculate the standard deviation of the lottery numbers
    return data[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']].std(axis=1)

def calculate_range(data):
    # Calculate the range (max - min) of the lottery numbers
    return data[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']].max(axis=1) - data[
        ['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']].min(axis=1)

def sum_numbers(data):
    # Calculate the sum of the lottery numbers
    return data[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']].sum(axis=1)

def odd_even_ratio(data):
    # Calculate the ratio of odd to even numbers
    odd_count = (data[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']] % 2).sum(axis=1)
    even_count = 5 - odd_count
    return odd_count / even_count

# Lists for pairs and impairs
pairs = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50]
impairs = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41, 43, 45, 47, 49]



In [ ]:
# Select only the columns with lottery numbers and chance number
df = df[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']]

In [ ]:
df

In [ ]:
# Scrape the data
df_tirage = pd.read_csv('/kaggle/input/loto-2008-2025/loto_2008_2025.csv')
# Reverse the DataFrame to have the most recent data last
df = df_tirage.iloc[::-1]
# Select only the columns with lottery numbers and chance number
df = df[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']]

In [ ]:
df

In [ ]:
# Apply feature engineering
df['freq_num0'] = freq_val(df, 'boule_1')
df['freq_num1'] = freq_val(df, 'boule_2')
df['freq_num2'] = freq_val(df, 'boule_3')
df['freq_num3'] = freq_val(df, 'boule_4')
df['freq_num4'] = freq_val(df, 'boule_5')
#df['freq_chance'] = freq_val(df, 'chance')
df['sum_diff'] = sum_diff(df)
#df['pair_chance'] = is_pair_etoile(df)
#df['impair_chance'] = is_impair_etoile(df)
df['pair'] = is_pair(df)
df['impair'] = is_impair(df)
df['is_under_24'] = is_under(df, 24)
df['is_under_40'] = is_under(df, 40)
df['mean'] = calculate_mean(df)
df['median'] = calculate_median(df)
df['std'] = calculate_std(df)
df['range'] = calculate_range(df)
df['sum'] = sum_numbers(df)
df['odd_even_ratio'] = odd_even_ratio(df)

In [ ]:
df = pd.read_csv('/kaggle/input/loto-2008-2025/loto_2008_2025.csv')

In [ ]:
df = df[['boule_2']]

In [ ]:
df

In [ ]:
def calculate_mean(data):
    # Calculate the mean of the lottery numbers
    #return data[['boule_1', 'boule_2', 'boule_3', 'boule_4', 'boule_5']].mean(axis=1)
    return data[['boule_1']].mean(axis=1)

In [ ]:
df['mean'] = calculate_mean(df)

In [ ]:
mean_value = df['boule_2'].mean()

In [ ]:
mean_value

In [ ]:
# Remove any infinite or excessively large values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)


In [ ]:
# Model parameters
nb_label_feature = 6
UNITS = 100
BATCHSIZE = 30
EPOCH = 1000
OPTIMIZER = 'adam'
LOSS = 'mae'
DROPOUT = 0.1
window_length = 12

In [ ]:
# Define LSTM model
def define_model(number_of_features, nb_label_feature):
    model = Sequential()
    model.add(LSTM(UNITS, input_shape=(window_length, number_of_features), return_sequences=True))
    model.add(LSTM(UNITS, dropout=0.1, return_sequences=False))
    model.add(Dense(nb_label_feature))
    model.compile(loss=LOSS, optimizer=OPTIMIZER, metrics=['acc'])
    return model


In [ ]:
# Create dataset for LSTM
def create_lstm_dataset(df, window_length, nb_label_feature):
    number_of_rows = df.shape[0]
    number_of_features = df.shape[1]
    # Standardize the dataset
    scaler = StandardScaler().fit(df.values)
    transformed_dataset = scaler.transform(df.values)
    transformed_df = pd.DataFrame(data=transformed_dataset, index=df.index)

    # Initialize arrays for training data and labels
    train = np.empty([number_of_rows - window_length, window_length, number_of_features], dtype=float)
    label = np.empty([number_of_rows - window_length, nb_label_feature], dtype=float)
    # Create the LSTM dataset
    for i in range(0, number_of_rows - window_length):
        train[i] = transformed_df.iloc[i:i + window_length, 0: number_of_features].values
        label[i] = transformed_df.iloc[i + window_length: i + window_length + 1, 0:nb_label_feature].values

    return train, label, scaler

# preprocessed DataFrame
train, label, scaler1 = create_lstm_dataset(df, window_length, nb_label_feature)

# Check if a saved model exists
#try:
    #best_model = load_model('best_model.keras')
    #print("Loaded existing model.")
#except:
 #   best_model = define_model(train.shape[2], nb_label_feature)
  #  print("No existing model found. Created a new model.")


In [ ]:
# Define checkpoint callback
checkpoint_callback = ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)


In [ ]:
# Train the model with adjusted EarlyStopping
history = best_model.fit(
    train,
    label,
    batch_size=30,
    epochs=1000,
    verbose=2,
    validation_split=0.2,  # Use a validation split to monitor val_loss
    callbacks=[EarlyStopping(monitor='val_loss', mode='min', patience=200), checkpoint_callback]
#   Consider improving patience and compare...
#   callbacks=[EarlyStopping(monitor='val_loss', mode='min', patience=400), checkpoint_callback]
)

In [ ]:
# Plot training loss
plt.plot(history.history['loss'])
plt.legend(['train_loss'])
plt.show()

# Make predictions
last_twelve = df.tail(window_length)
scaled_to_predict = scaler1.transform(last_twelve.values)

scaled_predicted_output = best_model.predict(np.array([scaled_to_predict]))

# Create a placeholder for all features and fill it with predictions
placeholder = np.zeros((1, df.shape[1]))
placeholder[0, :6] = scaled_predicted_output

# Inverse transform the placeholder without feature names
original_scale_pred = scaler1.inverse_transform(placeholder)

# Print only the first 6 elements (predictions)
print(original_scale_pred[0, :6].astype(int))

In [ ]:
import math
import pandas_datareader as web
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM
import matplotlib.pyplot as plt

In [ ]:

# Data Frame did import from google colab. 
# You can upload the data frame on your google drive and mounting the drive with the google clab
# df = web.DataReader('BTC-USD', data_source='yahoo', start='2014-01-01', end='2022-01-10')
df=pd.read_csv('/kaggle/input/loto-2008-2025/loto_2008_2025.csv')
df.head()

In [ ]:
df = df.set_index('date_de_tirage')

In [ ]:
# pour afficher entre deux dates precises

df= df.loc['2008-10-06':'2009-10-13']

In [ ]:
df

In [ ]:


plt.figure(figsize=(16,8))
plt.title('Boule 1', fontsize=24)
plt.plot(df['boule_1'])
plt.xlabel('dates', fontsize=18)
plt.ylabel('numeros', fontsize=18)
plt.show()
     


In [ ]:
#Creat a new dataframe with only Close Price
data = df.filter(['boule_1'])
#Convert the dataframe to numpy array
dataset = data.values
# Get the number of rows to train the model on. we need this number to create our train and test sets
# math.ceil will round up the number
training_data_len = math.ceil(len(dataset) * .8) # We are using %80 of the data for training
training_data_len

In [ ]:
# Scale the data
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(dataset)
scaled_data

In [ ]:
# Creat the training dataset
train_data = scaled_data[0:training_data_len, :]
# Split the data into X_train and y_train data sets
X_train = []
y_train = []

for i in range(60, len(train_data)):
  X_train.append(train_data[i-60: i, 0])
  y_train.append(train_data[i, 0])


  if i <= 60:
    print(X_train)
    print(y_train)
    print()

In [ ]:


len(X_train)
     


In [ ]:

# Convert the X_train and y_train to numpy array
X_train, y_train = np.array(X_train), np.array(y_train)

In [ ]:
X_train.shape

In [ ]:
# Reshape the data because LSTM needs 3 dim
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1)) # we make it like pros. it wanna say "np.reshape(X_train, (2083, 60, 1))"
X_train.shape

In [ ]:


# Create the testing dataset
# Create a new array containing scaled values from index 2083
test_data = scaled_data[training_data_len - 60 : , :]

#Create the data sets X_test and y_test
X_test = []
y_test = dataset[training_data_len : , :]
for i in range(60, len(test_data)):
  X_test.append(test_data[i-60 : i, 0])

     


In [ ]:


# Convert the data to a numpy array 
X_test = np.array(X_test)
     


In [ ]:

# Reshape the test data
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

In [ ]:

# Build LSTM model
tf.random.set_seed(42)

model_1 = Sequential()
model_1.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1))) # we made it like pros ;) , the simple form is "input_shape(60, 1)""
model_1.add(LSTM(50, return_sequences=False))
model_1.add(Dense(25))
model_1.add(Dense(1))

In [ ]:


# Compile the model
model_1.compile(optimizer='adam', loss='mse')
     


In [ ]:


# Train the model
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=2)
history = model_1.fit(X_train, y_train, batch_size=1, epochs=10)
     


In [ ]:
# Get the model_1 predicted price values 
predictions_1 = model_1.predict(X_test)
predictions_1 = scaler.inverse_transform(predictions_1)
len(predictions_1)

In [ ]:
# Get the root mean squared error (RMSE) for model_1
rmse = np.sqrt(np.mean(predictions_1 - y_test)**2)
rmse

In [ ]:
# Let's plot the history of model_1 and see what's going on
historyForPlot = pd.DataFrame(history.history)
historyForPlot.index += 1 # we plus 1 to the number of indexing so our epochs Plot picture will be counting from 1 not 0.
historyForPlot.plot()
plt.ylabel("loss")
plt.xlabel("epochs")

In [ ]:


# Train the model again with 7 epochs
# but first we need to create another model so we can compare them together

# building LSTM model_2
tf.random.set_seed(42)

model_2 = Sequential()
model_2.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1))) # we made it like pros ;) | the simple form is "input_shape(60, 1)
model_2.add(LSTM(50, return_sequences=False))
model_2.add(Dense(25))
model_2.add(Dense(1))

# Compile model_2
model_2.compile(optimizer='Adam', loss='mse')

# Fit model_2
history_2 = model_2.fit(X_train, y_train, batch_size=1, epochs=20)
     


In [ ]:
# Get the model_2 predicted price values 
predictions_2 = model_2.predict(X_test)
predictions_2 = scaler.inverse_transform(predictions_2)
len(predictions_2)

In [ ]:


# Get the root mean squared error (RMSE) for model_2
rmse_2 = np.sqrt(np.mean(predictions_2 - y_test)**2)
rmse_2
     


In [ ]:
# Plot the data
train = data[:training_data_len]

#data for model_1
valid_1 = data[training_data_len:]
valid_1['Predictions'] = predictions_1

# data for model_2
valid_2 = data[training_data_len:]
valid_2['Predictions'] = predictions_2

# Visualized the data 
#model_1
plt.figure(figsize=(14, 10))
plt.subplot(2, 1, 1)
plt.title('Model_1 with 10 epochs')
plt.xlabel('Data', fontsize=18)
plt.ylabel('numeros', fontsize=18)
plt.plot(train['boule_1'])
plt.plot(valid_1[['boule_1', 'Predictions']])

plt.legend(['Train', 'Valid', 'Predictions'], loc='upper left')

#model_2
plt.subplot(2, 1, 2)
plt.title('Model_2 with 6 epochs')
plt.xlabel('Data', fontsize=18)
plt.ylabel('Close Price USD', fontsize=18)
plt.plot(train['boule_1'])
plt.plot(valid_2[['boule_1', 'Predictions']])

plt.legend(['Train', 'Valid', 'Predictions'], loc='upper left')

plt.subplots_adjust(left=0.1,
                    bottom=0.1, 
                    right=0.9, 
                    top=1, 
                    wspace=0.4, 
                    hspace=0.4)
plt.show()

In [ ]:


#Get the last 60 day closing price values and convert the datadrame to an array
last_60_days = data[-60:].values
# Scale the data to be values between 0 and 1
last_60_days_scaled = scaler.fit_transform(last_60_days)
# create an empty list
new_X_test = []
# Append the past 60 days
new_X_test.append(last_60_days_scaled)
# Convert the X_test data set to a numpy array
new_X_test = np.array(new_X_test)
# Reshape the data 
new_X_test = np.reshape(new_X_test, (new_X_test.shape[0], new_X_test.shape[1], 1))
# Get the predicted scaled price
pred_price = model_1.predict(new_X_test)
# Undo the scaling
pred_price = scaler.inverse_transform(pred_price)
print(pred_price)
     


In [ ]:
# Importing required packages
import numpy
import matplotlib.pyplot as plt
import pandas
import math
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
# fix random seed for reproducibility
numpy.random.seed(7)

In [ ]:
# load the dataset
dataframe = pd.read_csv('/kaggle/input/loto-2008-2025/loto_2008_2025.csv')
#dataset = dataframe.values
#dataset = dataset.astype('float32')
# sélection ou extractions des colonnes à  traiter



In [ ]:
dataframe

In [ ]:

dataframe = dataframe[['boule_1']]

In [ ]:
dataframe

In [ ]:
dataset = dataframe.values

In [ ]:
#normalize the dataset
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(dataset)
dataframe.head()

In [ ]:


# split into train and test sets
train_size = int(len(dataset) * 0.67)
test_size = len(dataset) - train_size
train, test = dataset[0:train_size,:], dataset[train_size:len(dataset),:]
print(len(train), len(test))
     


In [ ]:


# convert an array of values into a dataset matrix
def create_dataset(dataset, look_back=1):
	dataX, dataY = [], []
	for i in range(len(dataset)-look_back-1):
		a = dataset[i:(i+look_back), 0]
		dataX.append(a)
		dataY.append(dataset[i + look_back, 0])
	return numpy.array(dataX), numpy.array(dataY)
     

# reshape into X=t and Y=t+1
look_back = 1
trainX, trainY = create_dataset(train, look_back)
testX, testY = create_dataset(test, look_back)
# reshape input to be [samples, time steps, features]
trainX = numpy.reshape(trainX, (trainX.shape[0], 1, trainX.shape[1]))
testX = numpy.reshape(testX, (testX.shape[0], 1, testX.shape[1]))
     


In [ ]:


# make predictions
trainPredict = model.predict(trainX)
testPredict = model.predict(testX)
# invert predictions
trainPredict = scaler.inverse_transform(trainPredict)
trainY = scaler.inverse_transform([trainY])
testPredict = scaler.inverse_transform(testPredict)
testY = scaler.inverse_transform([testY])
# calculate root mean squared error
trainScore = math.sqrt(mean_squared_error(trainY[0], trainPredict[:,0]))
testScore = math.sqrt(mean_squared_error(testY[0], testPredict[:,0]))

# shift train predictions for plotting
trainPredictPlot = numpy.empty_like(dataset)
trainPredictPlot[:, :] = numpy.nan
trainPredictPlot[look_back:len(trainPredict)+look_back, :] = trainPredict
# shift test predictions for plotting
testPredictPlot = numpy.empty_like(dataset)
testPredictPlot[:, :] = numpy.nan
testPredictPlot[len(trainPredict)+(look_back*2)+1:len(dataset)-1, :] = testPredict
# plot baseline and predictions
plt.plot(scaler.inverse_transform(dataset))
plt.plot(trainPredictPlot)
plt.plot(testPredictPlot)
plt.show()
     


In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
     


In [ ]:
df = pd.read_csv('/kaggle/input/loto-2008-2025/loto_2008_2025.csv')

In [ ]:
df.head()

In [ ]:
df = df.set_index('date_de_tirage')

In [ ]:
df = df[['boule_1']]

In [ ]:
df

In [ ]:
df.plot(figsize=(12,6))

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
results = seasonal_decompose(df['boule_1'])
results.plot()

In [ ]:
len(df)

In [ ]:
train = df.iloc[:1000]
test = df.iloc[1000:]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

In [ ]:
df.head(),df.tail()

In [ ]:
scaler.fit(train)
scaled_train = scaler.transform(train)
scaled_test = scaler.transform(test)

In [ ]:
scaled_train[:10]

In [ ]:


from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator


In [ ]:


# define generator
n_input = 3
n_features = 1
generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=1)
     


In [ ]:

X,y = generator[0]
print(f'Given the Array: \n{X.flatten()}')
print(f'Predict this y: \n {y}')

In [ ]:


X.shape
     


In [ ]:

# We do the same thing, but now instead for 12 months
n_input = 12
generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=1)

In [ ]:

from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM

In [ ]:


# define model
model = Sequential()
model.add(LSTM(100, activation='relu', input_shape=(n_input, n_features)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
     


In [ ]:


model.summary()
     


In [ ]:


# fit model
model.fit(generator,epochs=50)
     


In [ ]:


loss_per_epoch = model.history.history['loss']
plt.plot(range(len(loss_per_epoch)),loss_per_epoch)
     


In [ ]:


last_train_batch = scaled_train[-12:]
     


In [ ]:



last_train_batch = last_train_batch.reshape((1, n_input, n_features))
     


In [ ]:
model.predict(last_train_batch)

In [ ]:


scaled_test[0]
     


In [ ]:
test_predictions = []

first_eval_batch = scaled_train[-n_input:]
current_batch = first_eval_batch.reshape((1, n_input, n_features))

for i in range(len(test)):
    
    # get the prediction value for the first batch
    current_pred = model.predict(current_batch)[0]
    
    # append the prediction into the array
    test_predictions.append(current_pred) 
    
    # use the prediction to update the batch and remove the first value
    current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)

In [ ]:
test_predictions

In [ ]:
test.head()

In [ ]:

true_predictions = scaler.inverse_transform(test_predictions)
     

In [ ]:



test['Predictions'] = true_predictions
     

     

In [ ]:


test.plot(figsize=(14,5))
     


In [ ]:


from sklearn.metrics import mean_squared_error
from math import sqrt
rmse=sqrt(mean_squared_error(test['boule_1'],test['Predictions']))
print(rmse)
     


# Loto chance lstm

###  d abord je charge les bibliotheques necessaires

In [ ]:
from tensorflow.keras.models import load_model
import tensorflow as tf
import math
import pandas_datareader as web
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout
from keras.layers import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping

plt.style.use('fivethirtyeight')



### maintenant je vais charger le dataframe, c'est a dire le fichier csv qui m'interesse

In [ ]:
# 
df = pd.read_csv('/kaggle/input/loto-2008-2025/loto_2008_2025(2).csv')
print('Number of rows and columns: ', df.shape)
print(df.head())
print("pour voir si des valeurs NAN sont presentes\n", df.isna().sum())

In [ ]:
df = df.set_index('date_de_tirage')
df

### Cette ligne de code est utilisée pour définir la colonne 'date_de_tirage' comme index du DataFrame
### Fonction principale :
### Transforme une colonne existante (ici 'date_de_tirage') en index (étiquettes de lignes) du DataFrame.
### Effets :
### La colonne 'date_de_tirage' devient l'index.
### Elle disparaît des colonnes "normales" du DataFrame (sauf si drop=False est spécifié).

In [ ]:
plt.figure(figsize = (14,10))
plt.plot(df["boule_1"])
plt.plot(df["boule_2"])
plt.plot(df["boule_3"])
plt.plot(df["boule_4"])
plt.plot(df['boule_5'])
plt.plot(df['numero_chance'])
plt.title('loto')
plt.ylabel('numeros')
plt.xlabel('quantite')
plt.legend(['boule_1','boule_2','boule_3','boule_4','boule_5'], loc='upper left')
plt.show()

In [ ]:
'''

# Create a dataframe with only the Close Stock Price Column
data_target = df.filter(['boule_1'])

# Convert the dataframe to a numpy array to train the LSTM model
target = data_target.values

# Splitting the dataset into training and test
# Target Variable: Close stock price value

training_data_len = math.ceil(len(target)* 0.75) # training set has 75% of the data
training_data_len

# Normalizing data before model fitting using MinMaxScaler
# Feature Scaling

sc = MinMaxScaler(feature_range=(0,1))
training_scaled_data = sc.fit_transform(target)
training_scaled_data

'''

In [ ]:
# .filter() : Une méthode pandas pour sélectionner des colonnes spécifiques
# 'boule_1' est la variable cible (target), on l'isole pour l'entraînement

data_target = df.filter(['boule_2'])

In [ ]:
# 

data_target = df

In [ ]:
data_target.tail(10)

In [ ]:
# .filter() : Une méthode pandas pour sélectionner des colonnes spécifiques
# 'numero_chance', est la variable cible (target), on l'isole pour l'entraînement

data_target = df.filter(['numero_chance'])
data_target.tail(10)

In [ ]:
# Cette ligne de code convertit un DataFrame pandas (data_target) en un tableau NumPy (target)

target = data_target.values
target

In [ ]:

#  cette ligne de code est utilisée pour diviser un ensemble de données en deux parties,
# où 75 % des données sont utilisées pour l'entraînement.

training_data_len = math.ceil(len(target)* 0.75) 
print('nombre de donnees pour l entrainement :',training_data_len)

In [ ]:
target.shape

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardisation
# StandardScaler est une classe de scikit-learn 
# qui standardise les caractéristiques en soustrayant la moyenne et en divisant par l'écart type.

standard_scaler = StandardScaler()
training_scaled_data = standard_scaler.fit_transform(target)

In [ ]:
# Normalizing data before model fitting using MinMaxScaler
# Feature Scaling

#sc = MinMaxScaler(feature_range=(0,1))
#training_scaled_data = sc.fit_transform(target)
#training_scaled_data

# creation des sequences temporelles

In [ ]:
# pas de temps

pas=24

### Creation des sequences d entrainement sans normalisation

In [ ]:
# cette routine n utilise pas les donnes normalisees

train_data = target[0:training_data_len  , : ]

In [ ]:
# suite

X_train = []
y_train = []
for i in range(15, len(train_data)): # La boucle commence à l'index 180 (pour avoir assez de données passées) et va jusqu'à la fin des données.
    X_train.append(train_data[i-15:i, 0]) # Pour chaque i, on prend les 180 valeurs précédentes (train_data[i-180:i]) et on les ajoute à X_train.
                                           # [:, 0] signifie qu'on prend uniquement la première colonne (utile si train_data a plusieurs features).
    y_train.append(train_data[i, 0]) # y_train contient la valeur suivante (à l'index i) que le modèle doit apprendre à prédire.

X_train, y_train = np.array(X_train), np.array(y_train) # Cette ligne convertit les listes Python X_train et y_train en tableaux NumPy (numpy.array
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1)) # X_train.shape[0] : Nombre d'échantillons
                                                                       # X_train.shape[1] : Nombre de pas de temps
                                                                       # 1 : Nombre de features (1 colonne dans train_data[:, 0]).
print('Numbre d echantillons de pas de temps et de colonnes : ', X_train.shape)  # Cette ligne de code affiche les dimensions (shape) du tableau X_train

###  Creation des sequences d entrainement avec normalisation

In [ ]:
# sequences temporelles
# pour creer les sequences temporelles
# Ce code prépare des données pour un modèle de machine learning,
# pour une tâche de série temporelle.
# vous prépariez des séquences de données pour un modèle qui a besoin de regarder en arrière
# sur une certaine fenêtre de temps (ici, 180 pas de temps) pour faire une prédiction.
# X_train va contenir les séquences d'entrée, et
# y_train va contenir les valeurs cibles correspondantes.

# for i in range(180, len(train_data)): : Cette boucle itère sur les indices de train_data à 
# partir de 180 jusqu'à la fin du tableau. Cela signifie que pour chaque itération, vous regardez 
# en arrière sur les 180 pas de temps précédents.


# 0:training_data_len : Cela signifie que vous sélectionnez toutes les lignes à partir de 
# l'index 0 jusqu'à l'index training_data_len - 1
# vous prenez les training_data_len premières lignes du tableau.

train_data = training_scaled_data[0:training_data_len  , : ]

X_train = []
y_train = []
for i in range(pas, len(train_data)): # La boucle commence à l'index 180 (pour avoir assez de données passées) et va jusqu'à la fin des données.
    X_train.append(train_data[i-pas:i,:]) # Pour chaque i, on prend les 180 valeurs précédentes (train_data[i-180:i]) et on les ajoute à X_train.
                                           # [:, 0] signifie qu'on prend uniquement la première colonne (utile si train_data a plusieurs features).
    y_train.append(train_data[i,:]) # y_train contient la valeur suivante (à l'index i) que le modèle doit apprendre à prédire.

X_train, y_train = np.array(X_train), np.array(y_train) # Cette ligne convertit les listes Python X_train et y_train en tableaux NumPy (numpy.array
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1)) # X_train.shape[0] : Nombre d'échantillons
                                                                       # X_train.shape[1] : Nombre de pas de temps
                                                                       # 1 : Nombre de features (1 colonne dans train_data[:, 0]).
print('Numbre de lignes et de colonnes : ', X_train.shape)  # Cette ligne de code affiche les dimensions (shape) du tableau X_train

In [ ]:
print('nombre d echantillons :' , X_train.shape[0])
print('nombre de pas de temps :', X_train.shape[1])
print('nombre de caracteristiques ou colonnes :' , X_train.shape[2])

In [ ]:
# rroutine sans normalisation

test_data = training_scaled_data[training_data_len - 2: , : ]

X_test = []
y_test =  target[training_data_len : , : ]
for i in range(15,len(test_data)):
    X_test.append(test_data[i-15:i,0])

X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0],X_test.shape[1],1))
print('Nombre de sequences, de pas de temps et de caracteristiques : ', X_test.shape)

In [ ]:
# sequences tempoerelles
# Il extrait les 180 dernières observations des données d'entraînement normalisées 
# (training_scaled_data) pour :
# Créer un ensemble de test (test_data).
# Servir de base pour prédire les valeurs futures 
# training_data_len - 180:
# Sélectionne les 180 derniers points (de l'index training_data_len - 180 à la fin)
# [:, :] Prend toutes les colonnes (utile si vos données ont plusieurs features).

test_data = training_scaled_data[training_data_len - pas: , : ]

# X_test : Contiendra des séquences de 180 pas de temps (fenêtres glissantes) pour faire des prédictions.
# y_test : Contiendra les valeurs réelles correspondantes à prédire (pour évaluer le modèle).
# 
X_test = []

# target : les valeurs cibles (ce que vous voulez prédire).
# training_data_len : Longueur des données d'entraînement.
# 
y_test =  target[training_data_len : , : ]

# test_data : Vos données de test (déjà normalisées, shape (N, features)).
# range(180, len(test_data)) : Parcourt les indices de 180 à la fin.
# test_data[i-180:i, 0] : Pour chaque i, prend une fenêtre de 180 pas de temps et la première colonne ([:, 0]).

for i in range(pas,len(test_data)):
    X_test.append(test_data[i-pas:i,:])

# # Cette ligne convertit la liste Python X_test en un tableau NumPy,
X_test = np.array(X_test)

# np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
# Prend le tableau NumPy X_test (2D) et le transforme en un tableau 3D
# Nouvelle forme : (nombre_séquences, pas_de_temps, nombre_features)
# Le 1 final indique qu'il n'y a qu'une seule feature par pas de temps

# print('Number of rows and columns: ', X_test.shape)
# Affiche les dimensions du tableau redimensionné, Format typique : (nombre_séquences, 180, 1)

X_test = np.reshape(X_test, (X_test.shape[0],X_test.shape[1],1))
print('Nombre de sequences, de pas de temps et de caracteristiques : ', X_test.shape)

In [ ]:
print('nombre d echantillons de test :' , X_test.shape[0])
print('nombre de pas de temps de test:', X_test.shape[1])
print('nombre de caracteristiques ou colonnes de test :' , X_test.shape[2])

### Creation des sequences de test avec normalisation

In [ ]:
!pip install -U protobuf==3.8.0

In [ ]:


model = Sequential()

#
model.add(LSTM(units = 240, return_sequences = True, input_shape = (X_train.shape[1], 1)))
model.add(Dropout(0.2))

# 
model.add(LSTM(units = 240, return_sequences = True))
model.add(Dropout(0.2))

# 
model.add(LSTM(units = 240, return_sequences = True))
model.add(Dropout(0.2))

# 
model.add(LSTM(units = 240))
model.add(Dropout(0.2))

# 
model.add(Dense(units = 1))




In [ ]:
# 

model.compile(optimizer = 'adam', loss = 'mean_squared_error')


In [ ]:
# entrainement du model

model.fit(X_train, y_train, epochs = 600, batch_size = 128, verbose=0)

In [ ]:
#

model.save("boule_2.keras")

In [ ]:
model = tf.keras.models.load_model('/kaggle/input/toute_boules/keras/default/1/toutes_boule.keras')

In [ ]:
# Utilise le modèle entraîné (model) pour faire des prédictions sur les données de test (X_test)
# X_test doit être dans le même format que les données d'entraînement (shape 3D pour un LSTM

prediction_numero_chance = model.predict(X_test)

# Inverse la normalisation appliquée initialement aux données
# Utilise le scaler original (standard_scaler ou minmax_scaler) qui avait été ajusté sur les 
# données d'entraînement
# Ramène les prédictions à l'échelle originale 

predicted_stock_price = standard_scaler.inverse_transform(prediction_numero_chance)

In [ ]:
predicted_stock_price[0:10]

In [ ]:

train = data_target[:training_data_len]
valid = data_target[training_data_len:]
valid['Predictions'] = predicted_stock_price


In [ ]:


# data_target : Votre série temporelle complète 
# training_data_len : Le point de séparation entre entraînement et test
# train contient les données d'entraînement (historique)
# valid contient les données de test/validation (période à prédire)
# predicted_stock_price : Les prédictions que vous avez obtenues après:
# Prédiction avec le modèle (model.predict(X_test))
# Cela ajoute une nouvelle colonne 'Predictions' au DataFrame valid

train = data_target[:training_data_len]
valid = data_target[training_data_len:]
valid['Predictions'] = predicted_stock_price

# affichage

plt.figure(figsize=(10,5))
plt.title('Model')
plt.xlabel('Date', fontsize=8)
plt.ylabel('numeros loto', fontsize=12)
plt.plot(train[['boule_1']])
plt.plot(valid[['boule_1', 'Predictions']])
plt.legend(['Train', 'Val', 'Predictions'], loc='lower right')
plt.show()



In [ ]:
valid.tail(12)

zone d etude ci  dessous 


### la sequence suivante pour predire le prochain numero
### je prends les 10 derniers numeros et j'applique mon model sur ces données.

In [ ]:
# to_predict = data_target.tail(10) # sélectionne les x dernières lignes de votre DataFrame
prochain_numero = data_target.tail(24)

# to_predict=target

# scaled_to_predict = standard_scaler.transform(to_predict)
scaled_to_predict = standard_scaler.transform(prochain_numero)

# predictions

scaled_predicted_output_1 = model.predict(np.array([scaled_to_predict]))
data = standard_scaler.inverse_transform(scaled_predicted_output_1).astype(int)

# df_predict = pd.DataFrame(data)
prediction_prochain_numero = pd.DataFrame(data)

#df_predict = pd.DataFrame(data, columns=['boule_1'])
#df.to_csv(''+filename+'.csv', index=False)
prediction_prochain_numero
#print('prediction_prochain_numero :', prediction_prochain_numero)

fin zone d etude

In [ ]:
31-

In [ ]:
import math
import pandas_datareader as web
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM
import matplotlib.pyplot as plt

In [ ]:
from bs4 import BeautifulSoup
import math
from tensorflow.keras import layers
import tensorflow as tf
import keras
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn import preprocessing
from sklearn.model_selection import TimeSeriesSplit
from keras.models import Sequential
from keras.models import load_model
from keras.layers import LSTM, Dense,Dropout
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense, Bidirectional, TimeDistributed, RepeatVector, Flatten
from keras.callbacks import EarlyStopping

# Chargement du dataframe

In [ ]:
#
df = pd.read_csv('/kaggle/input/loto-2008-2025/loto_2008_2025(2).csv')
print('Number of lignes et de  colonnes: ', df.shape)
print(df.head())
print("pour voir si des valeurs NAN sont presentes\n", df.isna().sum())

In [ ]:
df = df.set_index('date_de_tirage')
df

In [ ]:
# je supprime le dernier tirage pour le predire
df.drop(df.index[-1], inplace=True)
df


# Traitement boule_1

In [ ]:
# .filter() : Une méthode pandas pour sélectionner des colonnes spécifiques
# 'numero_chance', est la variable cible (target), on l'isole pour l'entraînement

data_target = df.filter(['boule_2'])
data_target

# Création des unites et dizaines

In [ ]:

import pandas as pd

# Isoler les dizaines (division entière par 10)
data_target['Dizaines'] = data_target['boule_2'] // 10


# Isoler les unités (reste de la division par 10)
data_target['Unites'] = data_target['boule_2'] % 10


In [ ]:
data_target

In [ ]:
# selection des unités et des dizaines

unites = data_target.filter(['Unites'])
dizaines = data_target.filter(['Dizaines'])


In [ ]:
# convertit les unités et dizaine en un tableau NumPy (target)

target_unites = unites.values
target_dizaines = dizaines.values


# Division des données pour la partie unités

In [ ]:
# Données séquentielles (non mélangées)
training_data_len_unites = int(len(target_unites) * 0.75)  # 75% pour l'entraînement

# Division
train_data = target_unites[:training_data_len_unites]  # Premiers 75%
test_data = target_unites[training_data_len_unites:]    # 25% restants

print(f"Entraînement : {len(train_data)} échantillons ({len(train_data)/len(target_unites):.0%})")
print(f"Test : {len(test_data)} échantillons ({len(test_data)/len(target_unites):.0%})")

In [ ]:
# Données séquentielles (non mélangées)
training_data_len_dizaines = int(len(target_dizaines) * 0.75)  # 75% pour l'entraînement

# Division
train_data = target_dizaines[:training_data_len_dizaines]  # Premiers 75%
test_data = target_dizaines[training_data_len_dizaines:]    # 25% restants

print(f"Entraînement : {len(train_data)} échantillons ({len(train_data)/len(target_unites):.0%})")
print(f"Test : {len(test_data)} échantillons ({len(test_data)/len(target_unites):.0%})")

In [ ]:
# unités
#  cette ligne de code est utilisée pour diviser un ensemble de données en deux parties,
# où 75 % des données sont utilisées pour l'entraînement.

training_data_len_unites = math.ceil(len(target_unites)* 0.75)
print('nombre de donnees pour l entrainement :',training_data_len_unites)

# Division des données pour la partie dizaines

In [ ]:
# dizaines
#  cette ligne de code est utilisée pour diviser un ensemble de données en deux parties,
# où 75 % des données sont utilisées pour l'entraînement.

training_data_len_dizaines = math.ceil(len(target_dizaines)* 0.75)
print('nombre de donnees pour l entrainement :',training_data_len_dizaines)

# Normalisation des untés

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardisation
# StandardScaler est une classe de scikit-learn
# qui standardise les caractéristiques en soustrayant la moyenne et en divisant par l'écart type.

standard_scaler = StandardScaler()
training_scaled_data_unites = standard_scaler.fit_transform(target_unites)

# Normalisation des dizaines

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardisation
# StandardScaler est une classe de scikit-learn
# qui standardise les caractéristiques en soustrayant la moyenne et en divisant par l'écart type.

standard_scaler = StandardScaler()
training_scaled_data_dizaines = standard_scaler.fit_transform(target_dizaines)

# Pas de temps

In [ ]:
pas=100

# Séquences temporelles X_train des unités

In [ ]:
# séquences temporelles

train_data = training_scaled_data_unites[0:training_data_len_unites  , : ]

X_train = [] # une liste de séquences d'entraînement
y_train = [] # une liste de valeurs cibles

for i in range(pas, len(train_data)): # La boucle commence à l'index 15 (pour avoir assez de données passées) et va jusqu'à la fin des données.
    X_train.append(train_data[i-pas:i, :]) # Prend une fenêtre de 15 points
    y_train.append(train_data[i, :]) # La valeur suivante comme cible. c'est lui que le modèle doit apprendre à prédire.

X_train, y_train = np.array(X_train), np.array(y_train) # Cette ligne convertit les listes python X_train et y_train en tableaux NumPy (numpy.array

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1)) # X_train.shape[0] : Nombre d'échantillons
                                                                       # X_train.shape[1] : Nombre de pas de temps
                                                                       # 1 : Nombre de features (1 colonne dans train_data[:, 0]).


In [ ]:
print('nombre d echantillons :' , X_train.shape[0])
print('nombre de pas de temps :', X_train.shape[1])
print('nombre de caracteristiques ou colonnes :' , X_train.shape[2])

# Séquence temporelles X_train des dizaines

In [ ]:
# séquences temporelles

train_data = training_scaled_data_dizaines[0:training_data_len_dizaines , : ]

X_train = [] # une liste de séquences d'entraînement
y_train = [] # une liste de valeurs cibles

for i in range(pas, len(train_data)): # La boucle commence à l'index 15 (pour avoir assez de données passées) et va jusqu'à la fin des données.
    X_train.append(train_data[i-pas:i, :]) # Prend une fenêtre de 15 points
    y_train.append(train_data[i, :]) # La valeur suivante comme cible. c'est lui que le modèle doit apprendre à prédire.

X_train, y_train = np.array(X_train), np.array(y_train) # Cette ligne convertit les listes python X_train et y_train en tableaux NumPy (numpy.array

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1)) # X_train.shape[0] : Nombre d'échantillons
                                                                       # X_train.shape[1] : Nombre de pas de temps
                                                                       # 1 : Nombre de features (1 colonne dans train_data[:, 0]).


# Séquence temporelles X_test des unités

In [ ]:
# sequences tempoerelles
# (training_scaled_data) pour :
# Créer un ensemble de test (test_data).
# Servir de base pour prédire les valeurs futures
# training_data_len - X:
# Sélectionne les X derniers points (de l'index training_data_len - X à la fin)
# [:, :] Prend toutes les colonnes (utile si vos données ont plusieurs features).

test_data = training_scaled_data_unites[training_data_len_unites - pas: , : ]

# X_test : Contiendra des séquences de X pas de temps (fenêtres glissantes) pour faire des prédictions.
# y_test : Contiendra les valeurs réelles correspondantes à prédire (pour évaluer le modèle).
#
X_test = []

# target : les valeurs cibles (ce que vous voulez prédire).
# training_data_len : Longueur des données d'entraînement.
#
y_test =  target_unites[training_data_len_unites : , : ]

# test_data : Vos données de test (déjà normalisées, shape (N, features)).
# range(X, len(test_data)) : Parcourt les indices de X à la fin.
# test_data[i-X:i, 0] : Pour chaque i, prend une fenêtre de X pas de temps et la première colonne ([:, 0]).

for i in range(pas,len(test_data)):
    X_test.append(test_data[i-pas:i,:])

# # Cette ligne convertit la liste Python X_test en un tableau NumPy,
X_test = np.array(X_test)

# np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
# Prend le tableau NumPy X_test (2D) et le transforme en un tableau 3D
# Nouvelle forme : (nombre_séquences, pas_de_temps, nombre_features)
# Le 1 final indique qu'il n'y a qu'une seule feature par pas de temps

# print('Number of rows and columns: ', X_test.shape)
# Affiche les dimensions du tableau redimensionné, Format typique : (nombre_séquences, 180, 1)

X_test = np.reshape(X_test, (X_test.shape[0],X_test.shape[1],1))
print('Nombre de sequences, de pas de temps et de caracteristiques : ', X_test.shape)


In [ ]:
print('nombre d echantillons de test :' , X_test.shape[0])
print('nombre de pas de temps de test:', X_test.shape[1])
print('nombre de caracteristiques ou colonnes de test :' , X_test.shape[2])

# Séquences temporelles X_test des dizaines

In [ ]:
# sequences tempoerelles
# (training_scaled_data) pour :
# Créer un ensemble de test (test_data).
# Servir de base pour prédire les valeurs futures
# training_data_len - X:
# Sélectionne les X derniers points (de l'index training_data_len - X à la fin)
# [:, :] Prend toutes les colonnes (utile si vos données ont plusieurs features).

test_data = training_scaled_data_dizaines[training_data_len_dizaines - pas: , : ]

# X_test : Contiendra des séquences de X pas de temps (fenêtres glissantes) pour faire des prédictions.
# y_test : Contiendra les valeurs réelles correspondantes à prédire (pour évaluer le modèle).
#
X_test = []

# target : les valeurs cibles (ce que vous voulez prédire).
# training_data_len : Longueur des données d'entraînement.
#
y_test =  target_dizaines[training_data_len_dizaines : , : ]

# test_data : Vos données de test (déjà normalisées, shape (N, features)).
# range(X, len(test_data)) : Parcourt les indices de X à la fin.
# test_data[i-X:i, 0] : Pour chaque i, prend une fenêtre de X pas de temps et la première colonne ([:, 0]).

for i in range(pas,len(test_data)):
    X_test.append(test_data[i-pas:i,:])

# # Cette ligne convertit la liste Python X_test en un tableau NumPy,
X_test = np.array(X_test)

# np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
# Prend le tableau NumPy X_test (2D) et le transforme en un tableau 3D
# Nouvelle forme : (nombre_séquences, pas_de_temps, nombre_features)
# Le 1 final indique qu'il n'y a qu'une seule feature par pas de temps

# print('Number of rows and columns: ', X_test.shape)
# Affiche les dimensions du tableau redimensionné, Format typique : (nombre_séquences, 180, 1)

X_test = np.reshape(X_test, (X_test.shape[0],X_test.shape[1],1))
print('Nombre de sequences, de pas de temps et de caracteristiques : ', X_test.shape)


In [ ]:
model = Sequential()

model.add(LSTM(units = 240, return_sequences = True, input_shape = (X_train.shape[1], 1)))
model.add(Dropout(0.2))

model.add(LSTM(units = 240, return_sequences = True))
model.add(Dropout(0.2))

model.add(LSTM(units = 240, return_sequences = True))
model.add(Dropout(0.2))

model.add(LSTM(units = 240))
model.add(Dropout(0.2))

model.add(Dense(units = 1))


In [ ]:
# Compiling the RNN
model.compile(optimizer = 'RMSprop', loss = 'mean_squared_error')

# Entrainement du model

In [ ]:
# Fitting the RNN to the Training set
model.fit(X_train, y_train, epochs = 800, batch_size = 64, verbose=0)

In [ ]:
#

model.save("boule_2_dizaines.keras")

In [ ]:
model = tf.keras.models.load_model('/kaggle/input/boule_1_unites/keras/default/1/boule_1_unites.keras')

In [ ]:
# Maintenant j'Utilise le modèle entraîné (model) pour faire des prédictions sur les données de test (X_test)
# X_test doit être dans le même format que les données d'entraînement (shape 3D pour un LSTM

prediction_numero_chance = model.predict(X_test)

# ensuite j'Inverse la normalisation appliquée initialement aux données
# j'Utilise le scaler original (standard_scaler ou minmax_scaler) qui avait été ajusté sur les
# données d'entraînement
# cela ramène les prédictions à l'échelle originale

predicted_stock_price = standard_scaler.inverse_transform(prediction_numero_chance)

In [ ]:
train = data_target[:training_data_len_unites]
valid = data_target[training_data_len_unites:]
valid['Predictions'] = predicted_stock_price

In [ ]:
valid

In [ ]:
# (data_target)  la série temporelle complète
# (training_data_len :) Le point de séparation entre entraînement et test
# (train)  contient les données d'entraînement (historique)
# (valid) contient les données de test/validation (période à prédire)
# (predicted_stock_price) Les prédictions que nous avons obtenues après:
# Prédiction avec le modèle (model.predict(X_test))
# Cela ajoute une nouvelle colonne 'Predictions' au DataFrame valid

train = data_target[:training_data_len_unites]
valid = data_target[training_data_len_unites:]
valid['Predictions'] = predicted_stock_price

# affichage

plt.figure(figsize=(10,5))
plt.title('Model')
plt.xlabel('Date', fontsize=8)
plt.ylabel('numeros loto', fontsize=12)
plt.plot(train['Unites'])
plt.plot(valid[['Unites', 'Predictions']])
plt.legend(['Train', 'Val', 'Predictions'], loc='lower right')
plt.show()

In [ ]:
# pourcentage partiel

import pandas as pd
train = unites[:training_data_len_unites]
valid = unites[training_data_len_unites:]

df['Matching_partiel'] = abs(valid - predicted_stock_price) <= 1
pourcentage_partiel = df['Matching_partiel'].mean() * 100

# 1. Matching exact
df['Exact_Match'] = valid == predicted_stock_price
# Calcul des pourcentages
exact_accuracy = df['Exact_Match'].mean() * 100

print(f"Matching exact: {exact_accuracy:.2f}%")
print(f"Pourcentage à ±1 près: {pourcentage_partiel:.2f}%")

In [ ]:
exact_accuracy

In [ ]:
# to_predict = data_target.tail(10) # sélectionne les x dernières lignes de votre DataFrame
prochain_numero = unites.tail(10)
prochain_numero 

In [ ]:
# to_predict = data_target.tail(10) # sélectionne les x dernières lignes de votre DataFrame
prochain_numero = dizaines.tail(14)

# to_predict=target

# scaled_to_predict = standard_scaler.transform(to_predict)
scaled_to_predict = standard_scaler.transform(prochain_numero)

# predictions

scaled_predicted_output_1 = model.predict(np.array([scaled_to_predict]))
data = standard_scaler.inverse_transform(scaled_predicted_output_1).astype(int)

# df_predict = pd.DataFrame(data)
prediction_prochain_numero_unites = pd.DataFrame(data)

#df_predict = pd.DataFrame(data, columns=['boule_1'])
#df.to_csv(''+filename+'.csv', index=False)
prediction_prochain_numero_unites
#print('prediction_prochain_numero :', prediction_prochain_numero)